In [1]:
import sqlite3

conn = sqlite3.connect("crypto_analytics.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS crypto_prices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp_utc TEXT,
    coin TEXT,
    price_usd REAL,
    price_inr REAL,
    market_cap_usd REAL,
    vol_24h_usd REAL,
    change_24h_usd_pct REAL
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS fear_greed_index (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp_utc TEXT,
    fng_value INTEGER,
    fng_classification TEXT
)
""")

conn.commit()
print("Tables created")

Tables created


In [2]:
import requests
from datetime import datetime

def fetch_crypto_prices():
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": "bitcoin,ethereum",
        "vs_currencies": "usd,inr",
        "include_market_cap": "true",
        "include_24hr_vol": "true",
        "include_24hr_change": "true"
    }

    for attempt in range(1, 6):  # try 5 times
        r = requests.get(url, params=params, timeout=15)

        if r.status_code == 429:
            wait = attempt * 10
            print(f"⚠️ Rate limit hit (429). Waiting {wait}s then retrying...")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r.json()

    raise Exception("❌ CoinGecko API rate limit: retries exhausted")

def fetch_fear_greed():
    url = "https://api.alternative.me/fng/"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.json()

def ingest_once_sql():
    ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    prices = fetch_crypto_prices()
    sentiment = fetch_fear_greed()

    # insert prices
    for coin in ["bitcoin", "ethereum"]:
        cur.execute("""
        INSERT INTO crypto_prices(timestamp_utc, coin, price_usd, price_inr, market_cap_usd, vol_24h_usd, change_24h_usd_pct)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            ts,
            coin,
            prices[coin].get("usd"),
            prices[coin].get("inr"),
            prices[coin].get("usd_market_cap"),
            prices[coin].get("usd_24h_vol"),
            prices[coin].get("usd_24h_change")
        ))

    # sentiment insert
    fng = sentiment["data"][0]
    cur.execute("""
    INSERT INTO fear_greed_index(timestamp_utc, fng_value, fng_classification)
    VALUES (?, ?, ?)
    """, (
        ts,
        int(fng["value"]),
        fng["value_classification"]
    ))

    conn.commit()
    print(f"Inserted at {ts}")

ingest_once_sql()

C:\Users\shashwat\AppData\Local\Temp\ipykernel_29996\4249474406.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Inserted at 2026-01-25 11:32:49


In [3]:
import time

N_RUNS = 3
SLEEP_SECONDS = 10   #  300 

for i in range(1, N_RUNS + 1):
    print(f"\n Run {i}/{N_RUNS}")
    try:
        ingest_once_sql()
    except Exception as e:
        print(" Error:", e)

    if i < N_RUNS:
        print(f" Sleeping {SLEEP_SECONDS} seconds...")
        time.sleep(SLEEP_SECONDS)

print("\n Finished runs")


 Run 1/3


C:\Users\shashwat\AppData\Local\Temp\ipykernel_29996\4249474406.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Inserted at 2026-01-25 11:32:50
 Sleeping 10 seconds...

 Run 2/3
Inserted at 2026-01-25 11:33:01
 Sleeping 10 seconds...

 Run 3/3
Inserted at 2026-01-25 11:33:12

 Finished runs


In [4]:
import pandas as pd

prices_df = pd.read_sql_query("SELECT * FROM crypto_prices", conn)
sentiment_df = pd.read_sql_query("SELECT * FROM fear_greed_index", conn)

print(prices_df.shape, sentiment_df.shape)
prices_df.tail()

(24, 8) (12, 4)


,id,timestamp_utc,coin,price_usd,price_inr,market_cap_usd,vol_24h_usd,change_24h_usd_pct
19,20,2026-01-25 11:32:50,ethereum,2938.83,269176.0,3.546988e+11,1.021661e+10,-0.643816
20,21,2026-01-25 11:33:01,bitcoin,88539.00,8109556.0,1.768923e+12,1.971713e+10,-1.061502
21,22,2026-01-25 11:33:01,ethereum,2938.83,269176.0,3.546988e+11,1.021661e+10,-0.643816
22,23,2026-01-25 11:33:12,bitcoin,88539.00,8109556.0,1.768923e+12,1.971713e+10,-1.061502
23,24,2026-01-25 11:33:12,ethereum,2938.83,269176.0,3.546988e+11,1.021661e+10,-0.643816


In [5]:
!pip install mysql-connector-python pandas requests

In [6]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Oscar_mishra@29",
    database="crypto_intelligence"
)

cursor = conn.cursor()
print("Connected to MySQL successfully")

Connected to MySQL successfully


In [8]:
import requests
from datetime import datetime, timezone

def fetch_crypto_prices():
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": "bitcoin,ethereum",
        "vs_currencies": "usd,inr",
        "include_market_cap": "true",
        "include_24hr_vol": "true",
        "include_24hr_change": "true"
    }
    r = requests.get(url, params=params, timeout=15)

    # Optional: show rate limit issue clearly
    if r.status_code == 429:
        raise Exception("⚠️ CoinGecko Rate Limit (429). Wait 1–2 minutes and try again.")

    r.raise_for_status()
    return r.json()

def fetch_fear_greed():
    url = "https://api.alternative.me/fng/"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.json()

def ingest_once_mysql():
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    prices = fetch_crypto_prices()
    fng = fetch_fear_greed()["data"][0]

    # Insert Fear & Greed index
    cursor.execute("""
        INSERT IGNORE INTO fear_greed_index(timestamp_utc, fng_value, fng_classification)
        VALUES (%s, %s, %s)
    """, (ts, int(fng["value"]), fng["value_classification"]))

    # Insert BTC & ETH prices
    for coin in ["bitcoin", "ethereum"]:
        cursor.execute("""
            INSERT IGNORE INTO crypto_prices(
                timestamp_utc, coin, price_usd, price_inr, market_cap_usd, vol_24h_usd, change_24h_usd_pct
            )
            VALUES (%s,%s,%s,%s,%s,%s,%s)
        """, (
            ts,
            coin,
            prices[coin].get("usd"),
            prices[coin].get("inr"),
            prices[coin].get("usd_market_cap"),
            prices[coin].get("usd_24h_vol"),
            prices[coin].get("usd_24h_change")
        ))

    conn.commit()
    print(f"✅ Inserted live data at {ts}")

# Test run
ingest_once_mysql()

✅ Inserted live data at 2026-01-25 11:34:55


In [9]:
def generate_alerts():
    # latest sentiment
    cursor.execute("""
        SELECT timestamp_utc, fng_value, fng_classification
        FROM fear_greed_index
        ORDER BY timestamp_utc DESC
        LIMIT 1
    """)
    s_ts, fng_value, fng_class = cursor.fetchone()

    # latest prices
    cursor.execute("""
        SELECT timestamp_utc, coin, price_usd, change_24h_usd_pct
        FROM crypto_prices
        WHERE timestamp_utc = %s
    """, (s_ts,))
    rows = cursor.fetchall()

    for ts, coin, price_usd, change_24h in rows:
        # basic risk zone
        if fng_value <= 25:
            risk_zone = "HIGH"
            trade_signal = "BUY"
            reason = "Extreme Fear detected → possible rebound zone"
        elif fng_value >= 75:
            risk_zone = "HIGH"
            trade_signal = "SELL"
            reason = "Extreme Greed detected → possible correction risk"
        else:
            risk_zone = "MED"
            trade_signal = "HOLD"
            reason = "Neutral sentiment → no strong trade_signal"

        cursor.execute("""
            INSERT INTO market_alerts(
                timestamp_utc, coin, trade_signal, risk_zone, reason,
                price_usd, fng_value, fng_classification
            )
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        """, (ts, coin, trade_signal, risk_zone, reason, price_usd, fng_value, fng_class))

    conn.commit()
    print("Alerts generated")

generate_alerts()

Alerts generated


In [10]:
import time

N_RUNS = 12
SLEEP_SECONDS = 300  # after testing make 300

for i in range(1, N_RUNS + 1):
    print(f"\n🚀 Run {i}/{N_RUNS}")
    ingest_once_mysql()
    generate_alerts()

    if i < N_RUNS:
        print(f"⏳ Sleeping {SLEEP_SECONDS} seconds...")
        time.sleep(SLEEP_SECONDS)

print("\n✅ Finished live data collection runs")


🚀 Run 1/12
✅ Inserted live data at 2026-01-25 11:35:04
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 2/12
✅ Inserted live data at 2026-01-25 11:40:05
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 3/12
✅ Inserted live data at 2026-01-25 11:45:06
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 4/12
✅ Inserted live data at 2026-01-25 11:50:08
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 5/12
✅ Inserted live data at 2026-01-25 11:55:09
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 6/12
✅ Inserted live data at 2026-01-25 12:00:11
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 7/12
✅ Inserted live data at 2026-01-25 12:05:12
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 8/12
✅ Inserted live data at 2026-01-25 12:10:14
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 9/12
✅ Inserted live data at 2026-01-25 12:15:15
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 10/12
✅ Inserted live data at 2026-01-25 12:20:16
Alerts generated
⏳ Sleeping 300 seconds...

🚀 Run 11

In [ ]:
import pandas as pd

df_prices = pd.read_sql("SELECT * FROM crypto_prices ORDER BY timestamp_utc DESC LIMIT 10", conn)
df_fng = pd.read_sql("SELECT * FROM fear_greed_index ORDER BY timestamp_utc DESC LIMIT 10", conn)
df_alerts = pd.read_sql("SELECT * FROM market_alerts ORDER BY timestamp_utc DESC LIMIT 10", conn)

display(df_prices)
display(df_fng)
display(df_alerts)

In [ ]:
print("working")